<a href="https://colab.research.google.com/github/ericiortega/aipi590-xai-fall2025/blob/main/assignments/mechanistic_interpretability/mechanistic_interpretability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mechanistic Interpretability: Explaining a Tiny Brain

**Course:** AIPI 590 – Emerging Trends in Explainable AI (Fall 2025)  
**Professor:** Dr. Brinnae Bent  
**Author:** Eric Ortega Rodriguez  
**Case Domain:** Mechanistic Interpretability & Neural Understanding  
**Assigned Task:** Build, train, and interpret a tiny neural network to uncover what one neuron or component is “thinking.”  
**Deadline:** November 10, 2025 @ 11:30 AM  
**GitHub Repo:** [Assignment Folder](https://github.com/ericiortega/aipi590-xai-fall2025/tree/main/assignments/mechanistic_interpretability)  

## Introduction

In this notebook, I explore the concept of mechanistic interpretability. I will do this by training a tiny neural network on a simple task and analyzing how its internal components (like neurons, activations, and weights) represent underlying meaningful logic.

The goal is to identify one interpretable neuron or feature that exhibits a clear and explainable pattern.

In [4]:
# Importing libraries (Based on Dr. Bent's Colab Notebook)
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
import numpy as np

## Part 1 – Setup (Train My Own Tiny Model on a Tiny Task)

In [3]:
# Generate Toy Dataset (Dr. Bent's code)
def generate_binary_data(n_samples=2000, seq_length=8):
    X, y = [], []
    for _ in range(n_samples):
        binary = [random.choice([0, 1]) for _ in range(seq_length)]
        label = sum(binary)
        X.append(binary)
        y.append(label)
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

seq_length = 8
n_classes = seq_length + 1
X_train, y_train = generate_binary_data(2000, seq_length)
X_val, y_val = generate_binary_data(500, seq_length)


In [5]:
# Updated Model for Regression (From Dr. Bent)
class CountingMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)  # Output is a single scalar

    def forward(self, x):
        h = F.relu(self.fc1(x))
        out = self.fc2(h)
        return out, h  # for interpretability

# Instantiate model
model = CountingMLP(input_dim=seq_length, hidden_dim=10)
print(model)

# Convert targets to float (regression labels)
y_train_reg = y_train.float().unsqueeze(1)
y_val_reg = y_val.float().unsqueeze(1)

# Training loop with MSE loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

losses = []
for epoch in range(100):
    model.train()
    out, _ = model(X_train)
    loss = criterion(out, y_train_reg)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print("Final training loss:", losses[-1])

CountingMLP(
  (fc1): Linear(in_features=8, out_features=10, bias=True)
  (fc2): Linear(in_features=10, out_features=1, bias=True)
)
Final training loss: 0.1282375454902649


In [6]:
# Evaluation: Mean Absolute Error + Rounded Accuracy (From Dr. Bent)
model.eval()
with torch.no_grad():
    preds, _ = model(X_val)
    mae = torch.abs(preds - y_val_reg).mean().item()
    rounded_preds = torch.round(preds).squeeze().long()
    accuracy = (rounded_preds == y_val).float().mean().item()

print(f"Validation MAE: {mae:.2f}")
print(f"Rounded Accuracy: {accuracy:.2f}")

Validation MAE: 0.29
Rounded Accuracy: 0.82


## Part 2 – Explore

## Part 3 – Explain

## Part 4 – Reflect